# Inverted rotary pendulum — PID simulation

균일한 bar rotary pendulum의 upright 상태를 $\theta=0$으로 선형화하고, 입력 $u=\ddot\phi$에 대한 PID 안정화 예제를 계산합니다.

## 1. 모델의 출발점: torque와 angular acceleration
직선 운동의 $F=ma$에 대응하여 회전 운동은
$$\tau=J\alpha=J\ddot q$$
입니다. 균일한 bar pendulum의 질량을 $m$, 전체 길이를 $l$이라 하고 한쪽 끝을 pivot으로 두면
$$l_c=\frac l2,\qquad J_p=\frac13ml^2.$$
Rotary arm 각도는 $\phi$, motor 축에서 pendulum pivot까지의 거리는 $r$입니다.

완전한 비선형식에는 $\dot\phi^2$, $\dot\theta^2$, $\dot\phi\dot\theta$ 항이 있지만 여기서는 **각속도가 작다**고 가정하여 이 2차 속도항들을 무시합니다. 그러면 downward 기준 전역 pendulum 각도 $\vartheta$에 대해
$$\frac13ml^2\ddot\vartheta+\frac12mrl\cos\vartheta\,\ddot\phi+\frac12mgl\sin\vartheta=0.$$
$J_p=\frac13ml^2$로 나누면
$$\ddot\vartheta+\frac{3r}{2l}\cos\vartheta\,\ddot\phi+\frac{3g}{2l}\sin\vartheta=0.$$
여기서 질량 $m$은 소거됩니다. 즉 $u=\ddot\phi$를 이상적인 입력으로 주는 모델에서 pendulum 각도 응답은 $m$에 직접 의존하지 않습니다. 다만 실제 step motor가 그 각가속도를 만들기 위해 필요한 torque에는 $m$이 영향을 줍니다.

편의를 위해
$$a=\frac{3g}{2l},\qquad b=\frac{3r}{2l},\qquad u=\ddot\phi$$
로 둡니다.

## 2. Inverted 평형점 선형화와 transfer function
Upright 주변 local angle을 $\vartheta=\pi+\theta$로 정의하여 upright 자체를 $\theta=0$으로 둡니다. 작은 각도에서
$$\sin(\pi+\theta)\approx-\theta,\qquad \cos(\pi+\theta)\approx-1.$$
따라서
$$\boxed{\ddot\theta-a\theta=bu}$$
이고
$$\boxed{P_i(s)=\frac{\Theta(s)}{U(s)}=\frac{b}{s^2-a}}.$$
수치값을 넣으면
$$\boxed{P_i(s)=\frac{0.893617}{s^2-62.6170}}.$$
$p=+\sqrt{62.6170}\approx+7.913\,\mathrm{s^{-1}}$인 양의 실수 pole이 있으므로 open-loop upright는 불안정합니다.

## 3. PID 설계
Reference를 $\theta_{ref}=0$으로 두면 $e=-\theta$입니다.
$$u=K_pe+K_i\int e\,dt+K_d\dot e.$$
Plant $b/(s^2-a)$와 negative feedback을 구성하면 closed-loop characteristic polynomial은
$$\boxed{s^3+bK_ds^2+(bK_p-a)s+bK_i=0}.$$
예제로 원하는 pole을 $-4,-5,-6$으로 두고 계수를 비교해 gain을 계산합니다. 최소한 $s$ 계수를 양수로 만들려면
$$K_p>\frac ab=\frac gr\approx70.07$$
가 필요합니다. 실제 step motor를 고려하여 $|u|\le30\,\mathrm{rad/s^2}$ saturation도 넣습니다.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Plot text is intentionally English/ASCII so it renders correctly on any OS/Jupyter setup.
# This also fixes the Unicode minus issue if Korean labels are added later.
plt.rcParams["axes.unicode_minus"] = False

g=9.81
l=0.235
r=0.14
a=3*g/(2*l)
b=3*r/(2*l)
print(f"a={a:.6f} 1/s^2, b={b:.6f}")
desired=np.array([-4.,-5.,-6.])
coef=np.poly(desired)
Kd=coef[1]/b
Kp=(coef[2]+a)/b
Ki=coef[3]/b
print(f"Kp={Kp:.6f}, Ki={Ki:.6f}, Kd={Kd:.6f}")

In [ ]:
theta0=np.deg2rad(5.0); u_max=30.0; t_end=5.0
def rhs(t,x):
    theta,theta_dot,integ_e,phi,phi_dot=x
    e=-theta
    u=np.clip(Kp*e+Ki*integ_e-Kd*theta_dot,-u_max,u_max)
    return [theta_dot,a*theta+b*u,e,phi_dot,u]
t_eval=np.linspace(0,t_end,5001)
sol=solve_ivp(rhs,(0,t_end),[theta0,0,0,0,0],t_eval=t_eval,max_step=.001,rtol=1e-9,atol=1e-11)
t=sol.t; theta,theta_dot,integ_e,phi,phi_dot=sol.y
u=np.clip(Kp*(-theta)+Ki*integ_e-Kd*theta_dot,-u_max,u_max)
print(f"peak |u| = {np.max(np.abs(u)):.4f} rad/s^2")
print(f"final theta = {np.rad2deg(theta[-1]):.6f} deg")
print(f"final phi = {np.rad2deg(phi[-1]):.3f} deg")

In [ ]:
fig,ax=plt.subplots(4,1,figsize=(9,9),sharex=True)
ax[0].plot(t,np.rad2deg(theta)); ax[0].axhline(0,lw=.8); ax[0].set_ylabel("theta [deg]")
ax[1].plot(t,u); ax[1].axhline(u_max,ls="--",lw=.8); ax[1].axhline(-u_max,ls="--",lw=.8); ax[1].set_ylabel("arm accel [rad/s^2]")
ax[2].plot(t,phi_dot); ax[2].set_ylabel("phi_dot [rad/s]")
ax[3].plot(t,np.rad2deg(phi)); ax[3].set_ylabel("phi [deg]"); ax[3].set_xlabel("time [s]")
for axy in ax: axy.grid(True)
fig.suptitle("Inverted pendulum: PID stabilization from 5 deg")
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/inverted_pid_response.png",dpi=160,bbox_inches="tight")
plt.show()

## 4. 결과 해석과 한계
이 PID는 pendulum angle $\theta$를 0으로 안정화하지만 rotary arm position $\phi$를 원점으로 복귀시키는 조건은 없습니다. 따라서 balance가 되어도 arm이 다른 위치에 남을 수 있습니다. 실제 장비에서는 $\theta,\dot\theta,\phi,\dot\phi$를 함께 사용하는 LQR/full-state feedback이 더 자연스럽습니다. 또한 stepper torque-speed limit, missed step, friction, sampling delay를 포함하면 실제 응답과 더 가까워집니다.